# TabPFN v2 OOF — 6th stack member (GPU)

Computes 5-fold out-of-fold probabilities plus full-data test probabilities with
TabPFN v2 ([Hollmann et al., Nature 2025](https://www.nature.com/articles/s41586-024-08328-6)).
Outputs `oof_tabpfn.npy` and `test_tabpfn.npy` for the ensemble in the repo.
Requires internet (weight download) and a GPU accelerator.


In [ ]:
!pip -q install tabpfn==2.0.9
import numpy as np, pandas as pd, torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
from pathlib import Path

DATA = Path('/kaggle/input/spaceship-titanic-data')

train = pd.read_csv(DATA / 'train.csv'); test = pd.read_csv(DATA / 'test.csv')

y = train['Transported'].astype(int).values; n = len(train)

# same feature engineering as the repo's best-model notebook

exec(open('/kaggle/usr/lib/spaceship-titanic-best-model-0-800-lb' + '/__notebook_source__.py').read()) if False else None

def prep(tr, te):

    df = pd.concat([tr.drop(columns=['Transported']), te], ignore_index=True)

    for c in ['HomePlanet', 'Destination', 'Cabin', 'Name']: df[c] = df[c].replace('', np.nan)

    parts = df['Cabin'].str.split('/', expand=True)

    df['Deck'] = parts[0]; df['CabinNum'] = pd.to_numeric(parts[1], errors='coerce'); df['Side'] = parts[2]

    df['group_id'] = df['PassengerId'].str.split('_').str[0]

    df['GroupSize'] = df.groupby('group_id')['PassengerId'].transform('count')

    df['IsSolo'] = (df['GroupSize'] == 1).astype(int)

    df['surname'] = df['Name'].str.split(' ').str[1]

    df['FamilySize'] = df.groupby(['group_id', 'surname'], dropna=False)['PassengerId'].transform('count')

    df.loc[df['surname'].isna(), 'FamilySize'] = np.nan

    spend = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

    df['TotalSpend'] = df[spend].sum(axis=1, min_count=1)

    df['LogSpend'] = np.log1p(df['TotalSpend'])

    df['NoSpend'] = (df['TotalSpend'] == 0).astype(float); df.loc[df['TotalSpend'].isna(), 'NoSpend'] = np.nan

    df['SpendMissing'] = df['TotalSpend'].isna().astype(int)
    for c in spend:
        df[c + 'Zero'] = (df[c] == 0).astype(float)
        df.loc[df[c].isna(), c + 'Zero'] = np.nan

    df['Leisure'] = df[['Spa', 'VRDeck']].sum(axis=1, min_count=1)

    df['Essentials'] = df[['RoomService', 'FoodCourt']].sum(axis=1, min_count=1)

    df['CryoSleep'] = df['CryoSleep'].astype('boolean')

    df.loc[(df['TotalSpend'] > 0) & df['CryoSleep'].isna(), 'CryoSleep'] = False

    def gmode(s):

        m = s.dropna().mode(); return m.iloc[0] if len(m) else np.nan

    for c in ['HomePlanet', 'CryoSleep', 'Deck', 'Side', 'Destination']:

        df[c] = df.groupby('group_id')[c].transform(lambda s: s.fillna(gmode(s)))

    df['HomePlanet'] = df['HomePlanet'].fillna(df['Deck'].map({'A':'Europa','B':'Europa','C':'Europa','T':'Europa','G':'Earth'}))

    df['IsChild'] = np.where(df['Age'].isna(), np.nan, (df['Age'] < 13).astype(float))

    for c in ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side']: df[c] = df[c].astype('string')

    return df

fe = prep(train, test)

fe_tr, fe_te = fe.iloc[:n].reset_index(drop=True), fe.iloc[n:].reset_index(drop=True)

cat_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side']

num_cols = ['Age', 'CabinNum', 'CabinNumZ', 'GroupSize', 'IsSolo', 'FamilySize', 'TotalSpend',

            'LogSpend', 'NoSpend', 'SpendMissing', 'Leisure', 'Essentials', 'IsChild',

            'RoomServiceZero', 'FoodCourtZero', 'ShoppingMallZero', 'SpaZero', 'VRDeckZero']

# CabinNumZ needs Deck filled first

fe['CabinNumZ'] = fe.groupby('Deck')['CabinNum'].transform(lambda s: (s - s.mean()) / s.std(ddof=0) if s.notna().any() else s)

fe_tr, fe_te = fe.iloc[:n].reset_index(drop=True), fe.iloc[n:].reset_index(drop=True)

print('features ready', fe.shape)


In [ ]:
def tabpfn_frame(frame):
    out = frame[num_cols].astype(float).copy()
    for c in cat_cols:
        out[c] = pd.Categorical(frame[c].fillna('Missing'))
    return out

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from tabpfn import TabPFNClassifier

splits = list(StratifiedKFold(5, shuffle=True, random_state=100).split(np.zeros(n), y))
oof = np.zeros(n)
for i, (tr_i, va_i) in enumerate(splits):
    clf = TabPFNClassifier(device='cuda', ignore_pretraining_limits=True, random_state=42)
    clf.fit(tabpfn_frame(fe_tr).iloc[tr_i], y[tr_i])
    oof[va_i] = clf.predict_proba(tabpfn_frame(fe_tr).iloc[va_i])[:, 1]
    print(f'fold {i+1}/5: {accuracy_score(y[va_i], oof[va_i] > 0.5):.4f}', flush=True)
print('OOF tabpfn:', round(accuracy_score(y, oof > 0.5), 4))
np.save('oof_tabpfn.npy', oof)

In [ ]:
clf = TabPFNClassifier(device='cuda', ignore_pretraining_limits=True, random_state=42)
clf.fit(tabpfn_frame(fe_tr), y)
test_p = clf.predict_proba(tabpfn_frame(fe_te))[:, 1]
np.save('test_tabpfn.npy', test_p)
print('saved oof_tabpfn.npy / test_tabpfn.npy — DONE')